In [ ]:
from pathlib import Path
import os

RUN_MODE = "nominal"  # "nominal" | "disturb"
DISTURBANCE_PROFILE = "none"  # "none" | "ramp" | "fluctuation"
STATE_MODE = "standard"  # "standard" | "mismatch"
STYLE_PROFILE = "hybrid"  # "hybrid" | "paper" | "debug"
SAVE_PDF = False

DISTILLATION_DYN_PATH = None
DISTILLATION_SNAPS_PATH = None
DISTILLATION_VISIBLE = True


def resolve_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start] + list(start.parents):
        if (candidate / "utils").exists() and (candidate / "Data").exists():
            return candidate
    raise RuntimeError("Could not resolve the repo root from the current working directory.")


HERE = Path(os.getcwd())
REPO_ROOT = resolve_repo_root(HERE)


In [ ]:
import numpy as np
import torch

from DQN.dqn_agent import DQNAgent
from utils.helpers import apply_min_max, build_horizon_recipes
from utils.horizon_runner import run_dqn_mpc_horizon_supervisor
from utils.observer import compute_observer_gain
from utils.plotting import compare_mpc_rl_from_dirs, plot_horizon_results
from utils.rewards import make_reward_fn_relative_QR
from utils.state_features import get_rl_state_dim
from systems.distillation.config import (
    DELTA_T_HOURS,
    DISTILLATION_INPUT_BOUNDS,
    DISTILLATION_NOMINAL_CONDITIONS,
    DISTILLATION_RL_SETPOINTS_PHYS,
    DISTILLATION_SETPOINT_RANGE_PHYS,
    DISTILLATION_SS_INPUTS,
    HORIZON_CONTROL_GRID,
    HORIZON_PREDICT_GRID,
    RL_REWARD_DEFAULTS,
    default_plant_paths,
)
from systems.distillation.data_io import (
    canonical_baseline_path,
    copy_legacy_distillation_data,
    load_distillation_system_data,
    resolve_distillation_data_dir,
    resolve_distillation_result_dir,
)
from systems.distillation.labels import DISTILLATION_SYSTEM_METADATA
from systems.distillation.plant import build_distillation_system, distillation_system_stepper
from systems.distillation.scenarios import (
    build_distillation_disturbance_schedule,
    canonical_disturbance_profile,
    validate_run_profile,
)


In [ ]:
validate_run_profile(RUN_MODE, DISTURBANCE_PROFILE)
DISTURBANCE_PROFILE = canonical_disturbance_profile(RUN_MODE, DISTURBANCE_PROFILE)

RUN_PROFILE_MAP = {
    ("nominal", "none"): {
        "n_tests": 200,
        "set_points_len": 200,
        "warm_start": 2,
        "test_cycle": [False, False, False, False, False],
        "plot_start_episode": 1,
        "compare_start_episode": 1,
    },
    ("disturb", "ramp"): {
        "n_tests": 200,
        "set_points_len": 200,
        "warm_start": 2,
        "test_cycle": [False, False, False, False, False],
        "plot_start_episode": 1,
        "compare_start_episode": 1,
    },
    ("disturb", "fluctuation"): {
        "n_tests": 200,
        "set_points_len": 200,
        "warm_start": 2,
        "test_cycle": [False, False, False, False, False],
        "plot_start_episode": 1,
        "compare_start_episode": 1,
    },
}
RUN_PROFILE = RUN_PROFILE_MAP[(RUN_MODE, DISTURBANCE_PROFILE)]

copy_legacy_distillation_data(REPO_ROOT)
DATA_DIR = resolve_distillation_data_dir(REPO_ROOT)
RESULT_DIR = resolve_distillation_result_dir(REPO_ROOT)
DYN_PATH_DEFAULT, SNAPS_PATH_DEFAULT = default_plant_paths("horizon", DISTURBANCE_PROFILE)
DYN_PATH = Path(DISTILLATION_DYN_PATH).expanduser() if DISTILLATION_DYN_PATH else DYN_PATH_DEFAULT
SNAPS_PATH = Path(DISTILLATION_SNAPS_PATH).expanduser() if DISTILLATION_SNAPS_PATH else SNAPS_PATH_DEFAULT

nominal_conditions = DISTILLATION_NOMINAL_CONDITIONS.copy()
ss_inputs = DISTILLATION_SS_INPUTS.copy()
u_min = DISTILLATION_INPUT_BOUNDS["u_min"].copy()
u_max = DISTILLATION_INPUT_BOUNDS["u_max"].copy()
setpoint_y = DISTILLATION_SETPOINT_RANGE_PHYS.copy()

y_sp_scenario_phys = DISTILLATION_RL_SETPOINTS_PHYS.copy()
system = build_distillation_system(
    path=DYN_PATH,
    ss_inputs=ss_inputs,
    initialization_point=nominal_conditions,
    delta_t=DELTA_T_HOURS,
    visible=DISTILLATION_VISIBLE,
)
steady_states = {"ss_inputs": np.asarray(system.ss_inputs, float).copy(), "y_ss": np.asarray(system.y_ss, float).copy()}
system_data = load_distillation_system_data(REPO_ROOT, steady_states, setpoint_y, u_min, u_max)

A_aug = system_data["A_aug"]
B_aug = system_data["B_aug"]
C_aug = system_data["C_aug"]
data_min = system_data["data_min"]
data_max = system_data["data_max"]
min_max_dict = system_data["min_max_dict"]
inputs_number = int(B_aug.shape[1])
y_sp_scenario = apply_min_max(y_sp_scenario_phys, data_min[inputs_number:], data_max[inputs_number:])

RESULT_PREFIX = f"distillation_horizon_{RUN_MODE}_{DISTURBANCE_PROFILE}_{STATE_MODE}_unified"
COMPARE_PREFIX = f"distillation_compare_horizon_{RUN_MODE}_{DISTURBANCE_PROFILE}_{STATE_MODE}"
BASELINE_MPC_PATH = canonical_baseline_path(REPO_ROOT, RUN_MODE, DISTURBANCE_PROFILE)
TOTAL_STEPS = RUN_PROFILE["n_tests"] * RUN_PROFILE["set_points_len"] * len(y_sp_scenario_phys)
DISTURBANCE_SCHEDULE = build_distillation_disturbance_schedule(RUN_MODE, DISTURBANCE_PROFILE, TOTAL_STEPS)


In [ ]:
poles = np.array([0.032, 0.03501095, 0.04099389, 0.04190188, 0.07477281, 0.01153274, 0.41036367])
L = compute_observer_gain(A_aug, C_aug, poles)

PREDICT_GRID = HORIZON_PREDICT_GRID
CONTROL_GRID = HORIZON_CONTROL_GRID
HORIZON_RECIPES = build_horizon_recipes(PREDICT_GRID, CONTROL_GRID)
STATE_DIM = get_rl_state_dim(A_aug.shape[0], C_aug.shape[0], B_aug.shape[1], STATE_MODE)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
HIDDEN_LAYERS = [512, 512, 512, 512, 512]
BUFFER_SIZE = 40_000
DECISION_INTERVAL = 4

predict_h = 6
cont_h = 3
Q1_penalty = 1.0
Q2_penalty = 1.0
R1_penalty = 1.0
R2_penalty = 1.0

reward_params, reward_fn = make_reward_fn_relative_QR(
    data_min,
    data_max,
    n_inputs=2,
    **RL_REWARD_DEFAULTS,
)

nominal_qi = 0.0
nominal_qs = 0.0
nominal_hA = 0.0
qi_change = 1.0
qs_change = 1.0
ha_change = 1.0

n_tests = RUN_PROFILE["n_tests"]
set_points_len = RUN_PROFILE["set_points_len"]
warm_start = RUN_PROFILE["warm_start"]
TEST_CYCLE = RUN_PROFILE["test_cycle"]

print(f"Using device: {DEVICE}")
print(f"State dimension: {STATE_DIM}")
print(f"Number of horizon actions: {len(HORIZON_RECIPES)}")
print(reward_params)

dqn_agent = DQNAgent(
    state_dim=STATE_DIM,
    action_dim=len(HORIZON_RECIPES),
    hidden_dim=HIDDEN_LAYERS,
    gamma=0.99,
    lr=1e-4,
    batch_size=128,
    buffer_size=BUFFER_SIZE,
    grad_clip_norm=10.0,
    double_dqn=True,
    target_update="soft",
    tau=0.01,
    hard_update_interval=10_000,
    target_combine="q1",
    activation="relu",
    use_layer_norm=False,
    dropout=0.0,
    device=DEVICE,
    eps_start=0.3,
    eps_end=0.01,
    eps_decay_rate=0.99999,
    eps_decay_mode="exp",
)


In [ ]:
horizon_cfg = {
    "mode": RUN_MODE,
    "state_mode": STATE_MODE,
    "predict_h": predict_h,
    "cont_h": cont_h,
    "decision_interval": DECISION_INTERVAL,
    "warm_start": warm_start,
    "test_cycle": TEST_CYCLE,
    "n_tests": n_tests,
    "set_points_len": set_points_len,
    "nominal_qi": nominal_qi,
    "nominal_qs": nominal_qs,
    "nominal_ha": nominal_hA,
    "qi_change": qi_change,
    "qs_change": qs_change,
    "ha_change": ha_change,
    "b_min": system_data["b_min"],
    "b_max": system_data["b_max"],
    "Q1_penalty": Q1_penalty,
    "Q2_penalty": Q2_penalty,
    "R1_penalty": R1_penalty,
    "R2_penalty": R2_penalty,
    "reward_scale": 0.01,
}

runtime_ctx = {
    "system": system,
    "y_sp_scenario": y_sp_scenario,
    "steady_states": steady_states,
    "min_max_dict": min_max_dict,
    "agent": dqn_agent,
    "A_aug": A_aug,
    "B_aug": B_aug,
    "C_aug": C_aug,
    "L": L,
    "data_min": data_min,
    "data_max": data_max,
    "horizon_recipes": HORIZON_RECIPES,
    "reward_fn": reward_fn,
    "reward_params": reward_params,
    "system_stepper": distillation_system_stepper,
    "system_metadata": DISTILLATION_SYSTEM_METADATA,
    "disturbance_labels": DISTILLATION_SYSTEM_METADATA.get("disturbance_labels"),
    "disturbance_schedule": DISTURBANCE_SCHEDULE,
}

result_bundle = run_dqn_mpc_horizon_supervisor(horizon_cfg=horizon_cfg, runtime_ctx=runtime_ctx)
result_bundle["mpc_path_or_dir"] = BASELINE_MPC_PATH
result_bundle["reward_params"] = reward_params
result_bundle["run_profile"] = RUN_PROFILE

print(f"nFE: {result_bundle['nFE']}")
print(f"Avg reward samples: {len(result_bundle['avg_rewards'])}")


In [ ]:
out_dir_rl = plot_horizon_results(
    result_bundle=result_bundle,
    plot_cfg={
        "directory": DATA_DIR,
        "prefix_name": RESULT_PREFIX,
        "start_episode": RUN_PROFILE["plot_start_episode"],
        "recipe_counts": True,
        "save_pdf": SAVE_PDF,
        "style_profile": STYLE_PROFILE,
    },
)

out_dir_cmp = compare_mpc_rl_from_dirs(
    rl_dir=out_dir_rl,
    mpc_path_or_dir=BASELINE_MPC_PATH,
    reward_fn=reward_fn,
    directory=RESULT_DIR,
    prefix_name=COMPARE_PREFIX,
    compare_mode=RUN_MODE,
    start_episode=RUN_PROFILE["compare_start_episode"],
    save_pdf=SAVE_PDF,
    style_profile=STYLE_PROFILE,
)

print(out_dir_rl)
print(out_dir_cmp)

try:
    system.close(SNAPS_PATH)
except Exception:
    pass
